# PolaFusion — Inference & Ensemble Prediction

This notebook loads the trained models and generates final predictions.

**Pipeline:**
1. Load all 8 models (5× mDeBERTa + 3× XLM-R-large) for each subtask
2. Run soft-vote probability aggregation across all models
3. Apply fixed global thresholds (0.50 / 0.35 / 0.30)
4. Apply forced-argmax correction for Subtasks 2 & 3
5. Write final submission CSVs


In [ ]:
# ─── CONFIGURATION ───────────────────────────────────────────────────────────
import os, gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader

# ── Paths ──────────────────────────────────────────────────────────────────
TEST_DIR   = "/kaggle/input/semeval-task-9-p2"          # update as needed

# ── Subtask 1 models ───────────────────────────────────────────────────────
ST1_MDEB_MODELS = [f"/kaggle/input/mdeberta-st1/fold_{i}" for i in range(5)]
ST1_XLMR_MODELS = [f"/kaggle/input/xlmr-st1/fold_{i}"    for i in range(3)]

# ── Subtask 2 models ───────────────────────────────────────────────────────
ST2_MDEB_MODELS = [f"/kaggle/input/mdeberta-st2/fold_{i}" for i in range(5)]
ST2_XLMR_MODELS = [f"/kaggle/input/xlmr-st2/fold_{i}"    for i in range(3)]

# ── Subtask 3 models ───────────────────────────────────────────────────────
ST3_MDEB_MODELS = [f"/kaggle/input/mdeberta-st3/fold_{i}" for i in range(5)]
ST3_XLMR_MODELS = [f"/kaggle/input/xlmr-st3/fold_{i}"    for i in range(3)]

# ── Labels ─────────────────────────────────────────────────────────────────
ST1_LABELS = ["polarization"]
ST2_LABELS = ["political", "racial/ethnic", "religious", "gender/sexual", "other"]
ST3_LABELS = ["stereotype", "vilification", "dehumanization",
              "extreme_language", "lack_of_empathy", "invalidation"]

# ── Thresholds ─────────────────────────────────────────────────────────────
THRESH_ST1 = 0.50
THRESH_ST2 = 0.35
THRESH_ST3 = 0.30

MAX_LEN    = 128
BATCH_SIZE = 32
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# ─── DATASET ─────────────────────────────────────────────────────────────────
class InferenceDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts     = [str(t) for t in texts]
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt"
        )
        return {k: v.squeeze(0) for k, v in enc.items()}


In [ ]:
# ─── INFERENCE HELPER ────────────────────────────────────────────────────────
def run_model(model_path, texts, num_labels, batch_size=BATCH_SIZE):
    """Run one saved model on a list of texts; return sigmoid probabilities."""
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model     = AutoModelForSequenceClassification.from_pretrained(
        model_path, num_labels=num_labels
    ).to(DEVICE).eval()

    ds     = InferenceDataset(texts, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size)
    probs  = []

    with torch.no_grad():
        for batch in loader:
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch).logits
            probs.append(torch.sigmoid(logits).cpu().numpy())

    del model; torch.cuda.empty_cache(); gc.collect()
    return np.concatenate(probs, axis=0)


def soft_vote(model_paths, texts, num_labels):
    """Average sigmoid probabilities across all models (soft voting)."""
    all_probs = [run_model(p, texts, num_labels) for p in model_paths]
    return np.mean(all_probs, axis=0)


def forced_argmax(preds):
    """For rows where all labels are 0, set the highest-prob label to 1."""
    for i in range(len(preds)):
        if preds[i].sum() == 0:
            preds[i, np.argmax(preds[i])] = 1   # type: ignore
    return preds


In [ ]:
# ─── SUBTASK 1 INFERENCE ─────────────────────────────────────────────────────
import glob

st1_test_files = glob.glob(f"{TEST_DIR}/subtask1/test/*.csv")
st1_dfs = []

for fpath in st1_test_files:
    df = pd.read_csv(fpath)
    texts = df["text"].tolist()

    # Soft vote across 8 models
    probs = soft_vote(ST1_MDEB_MODELS + ST1_XLMR_MODELS, texts, num_labels=1)
    df["polarization"] = (probs[:, 0] > THRESH_ST1).astype(int)
    st1_dfs.append(df)

st1_out = pd.concat(st1_dfs, ignore_index=True)
st1_out.to_csv("subtask1_predictions.csv", index=False)
print(f"✅ Subtask 1 done — {len(st1_out)} rows")


In [ ]:
# ─── SUBTASK 2 INFERENCE ─────────────────────────────────────────────────────
# Only run on rows predicted as polarized in Subtask 1
polarized_df = st1_out[st1_out["polarization"] == 1].copy()
texts = polarized_df["text"].tolist()

probs = soft_vote(ST2_MDEB_MODELS + ST2_XLMR_MODELS, texts, num_labels=len(ST2_LABELS))
preds = (probs > THRESH_ST2).astype(int)

for i, col in enumerate(ST2_LABELS):
    polarized_df[col] = preds[:, i]

# Non-polarized rows get all-zero ST2 labels
non_pol = st1_out[st1_out["polarization"] == 0].copy()
for col in ST2_LABELS:
    non_pol[col] = 0

st2_out = pd.concat([polarized_df, non_pol], ignore_index=True)
st2_out.to_csv("subtask2_predictions.csv", index=False)
print(f"✅ Subtask 2 done — {len(st2_out)} rows")


In [ ]:
# ─── SUBTASK 3 INFERENCE ─────────────────────────────────────────────────────
# Re-use the same polarized subset
probs = soft_vote(ST3_MDEB_MODELS + ST3_XLMR_MODELS, texts, num_labels=len(ST3_LABELS))
preds = (probs > THRESH_ST3).astype(int)
preds = forced_argmax(preds)

for i, col in enumerate(ST3_LABELS):
    polarized_df[col] = preds[:, i]

non_pol3 = st1_out[st1_out["polarization"] == 0].copy()
for col in ST3_LABELS:
    non_pol3[col] = 0

st3_out = pd.concat([polarized_df, non_pol3], ignore_index=True)
st3_out.to_csv("subtask3_predictions.csv", index=False)
print(f"✅ Subtask 3 done — {len(st3_out)} rows")
print("\n🎉 All predictions saved!")
